In [1]:
import json
import os

import numpy as np
import pandas as pd
import torch

from evaluation_utils import MetricsForDatasetProbes, evaluate_classifier_performance
from phi_3_5_constants import directions_reconstruction_losses_path, dsets_index_path, dsets_folder, \
    finalized_activations_dir, hidden_state_size, four_way_topics_index_path, misc_datasets_index_path, \
    test_classification_metrics_path, analysis_results_folder
from phi_3_5_probe import ProbesForDataset, load_probes_for_dset

In [2]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")
num_dsets = dsets_index_df.shape[0]

In [3]:
all_dsets_activations: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]
all_dsets_labels: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]

In [4]:
# top-level key is the name of a topic that has pos/neg/conj/disj variants, second level key is one of those variant names
with four_way_topics_index_path.open("r") as f:
    dset_idxs_for_4way_topics: dict[str, dict[str, int]] = json.load(f)
with misc_datasets_index_path.open("r") as f:
    idxs_for_other_dsets: dict[str, int] = json.load(f)

In [5]:
directions_reconstruction_losses = pd.read_csv(directions_reconstruction_losses_path)
dir_probe_scenarios_strs = directions_reconstruction_losses['scenario_identifier'].to_list()
dir_probe_scenarios_specs = [tuple(map(int, scenario_str.split(' '))) for scenario_str in dir_probe_scenarios_strs]

In [6]:
probes: dict[tuple[int,...], ProbesForDataset] = {}
# first level key identifies the scenario (of one or more datasets) which the probe was trained on;
# second level key identifies the previously-unseen dataset which the probe is tested on
probes_metrics_on_test_dsets: dict[tuple[int,...], dict[int, MetricsForDatasetProbes]] = {}

In [7]:
for dset_idx, dset_dtls in dsets_index_df.iterrows():
    categ_nm = dset_dtls["Categ_Folder"]
    dset_file_nm = dset_dtls["Dataset_File"]
    dset_nm = os.path.splitext(dset_file_nm)[0]
    
    dataset = pd.read_csv(dsets_folder / categ_nm / dset_file_nm)
    dset_size = dataset.shape[0]
    dset_labels = torch.from_numpy(dataset['label'].to_numpy().astype(np.float32)[:, np.newaxis])
    all_dsets_labels[dset_idx] = dset_labels
    
    activs_path = finalized_activations_dir / categ_nm / (dset_nm + ".pt")
    relevant_activations = torch.load(activs_path, weights_only=True)
    assert relevant_activations.shape == (2, dset_size, hidden_state_size)
    all_dsets_activations[dset_idx] = relevant_activations
    if not dset_dtls['in_german']:
        # only using the german datasets for testing of generalization, so didn't learn directions for them or train probes with them
        curr_dset_probes = load_probes_for_dset(categ_nm, dset_nm, hidden_state_size)
        probes[(dset_idx,)] = curr_dset_probes

In [8]:
for topic_nm, variants_dset_idxs in dset_idxs_for_4way_topics.items():
    affirm_idx = variants_dset_idxs["affirm"]
    neg_idx = variants_dset_idxs["neg"]
    conj_idx = variants_dset_idxs["conj"]
    disj_idx = variants_dset_idxs["disj"]
    
    affirm_neg_key = tuple(sorted([affirm_idx, neg_idx]))
    affirm_neg_probes = load_probes_for_dset(topic_nm, "affirm_neg", hidden_state_size)
    probes[affirm_neg_key] = affirm_neg_probes
    
    affirm_disj_key = tuple(sorted([affirm_idx, disj_idx]))
    affirm_disj_probes = load_probes_for_dset(topic_nm, "affirm_disj", hidden_state_size)
    probes[affirm_disj_key] = affirm_disj_probes
    
    neg_conj_key = tuple(sorted([neg_idx, conj_idx]))
    neg_conj_probes = load_probes_for_dset(topic_nm, "neg_conj", hidden_state_size)
    probes[neg_conj_key] = neg_conj_probes
    
    affirm_neg_conj_disj_key = tuple(sorted([affirm_idx, neg_idx, conj_idx, disj_idx]))
    affirm_neg_conj_disj_probes = load_probes_for_dset(topic_nm, "affirm_neg_conj_disj", hidden_state_size)
    probes[affirm_neg_conj_disj_key] = affirm_neg_conj_disj_probes

In [9]:
unambig_lie_idx = idxs_for_other_dsets["unambiguous_lie"]
unambig_truth_idx = idxs_for_other_dsets["unambiguous_truthful_reply"]
ambig_truth_idx = idxs_for_other_dsets["ambiguous_truthful_reply"]
real_world_multi_dset_idxs = [unambig_lie_idx, unambig_truth_idx, ambig_truth_idx]
real_world_multi_dset_scenario_key = tuple(sorted(real_world_multi_dset_idxs))

real_world_multi_dset_probes = load_probes_for_dset("real_world_scenarios", "unambig_lie_unambig_truth_ambig_truth", hidden_state_size)
probes[real_world_multi_dset_scenario_key] = real_world_multi_dset_probes

In [10]:
multi_topics_dset_idxs = []
various_categories_dset_idxs = []

for topic_nm, variants_dset_idxs in dset_idxs_for_4way_topics.items():
    if topic_nm not in ("animal_class", "element_symbols", "facts", "inventors"):
        continue
    affirm_idx = variants_dset_idxs["affirm"]
    neg_idx = variants_dset_idxs["neg"]
    conj_idx = variants_dset_idxs["conj"]
    multi_topics_dset_idxs.extend([affirm_idx, neg_idx, conj_idx])
    various_categories_dset_idxs.extend([affirm_idx, neg_idx])

In [11]:
multi_topics_affirm_neg_conj_key = tuple(sorted(multi_topics_dset_idxs))
multi_topics_affirm_neg_conj_probes = load_probes_for_dset("", "multi_topics_affirm_neg_conj", hidden_state_size)
probes[multi_topics_affirm_neg_conj_key] = multi_topics_affirm_neg_conj_probes

In [12]:
smaller_than_idx = idxs_for_other_dsets["smaller_than"]
common_claim_idx = idxs_for_other_dsets["common_claim_true_false"]
various_categories_dset_idxs.extend(real_world_multi_dset_idxs)
various_categories_dset_idxs.extend([smaller_than_idx, common_claim_idx])
various_categories_scenario_key = tuple(sorted(various_categories_dset_idxs))
various_categories_probes = load_probes_for_dset("", "various_categories", hidden_state_size)
probes[various_categories_scenario_key] = various_categories_probes

In [13]:
for scenario_key in probes.keys():
    probes_metrics_on_test_dsets[scenario_key] = {}

In [14]:
for scenario_key, scenario_probes in probes.items():
    for test_dset_idx, test_dset_dtls in dsets_index_df.iterrows():
        if test_dset_idx in scenario_key:
            continue

        scenario_metrics_on_curr_test_dset = evaluate_classifier_performance(
            scenario_probes, all_dsets_activations[test_dset_idx], all_dsets_labels[test_dset_idx])
        probes_metrics_on_test_dsets[scenario_key][test_dset_idx] = scenario_metrics_on_curr_test_dset


In [15]:
analysis_results_folder.mkdir(exist_ok=True)

In [16]:
serialized_test_metrics = {
    (" ".join(map(str, scenario_id))): {
        str(test_dset_idx): test_dset_metrics.to_dict()
        for test_dset_idx, test_dset_metrics in test_dsets_metrics.items()
    } for scenario_id, test_dsets_metrics in probes_metrics_on_test_dsets.items()
}
with test_classification_metrics_path.open("w") as f:
    json.dump(serialized_test_metrics, f, indent=2)